# Project Manager Delegates to Child Agents

**Pattern:** Master agent delegates to child agents

A delivery manager coordinates a delayed software launch. Specialist agents assess schedule, quality, and customer communications.

Classroom simulation with fictional facts. Save an .env file beside this notebook with OPENAI_API_KEY=your_key. Install requirements.txt first.

In [ ]:
import os  # Read environment settings.
from dotenv import load_dotenv  # Load values from the .env file.
from langchain_openai import ChatOpenAI  # Use LangChain to call an OpenAI chat model.

load_dotenv()  # Load OPENAI_API_KEY from .env.
model = ChatOpenAI(model="gpt-4.1-mini", temperature=0)  # Share one model connection across role prompts.

def ask_agent(role, task, context):
    prompt = f"You are {role}. {task}\n\nCase facts and prior reports:\n{context}"  # Give this agent its role and handoff context.
    return model.invoke(prompt).content  # Return the agent's message.


## Synthetic case

Edit the sample details to explore how the agents respond.

In [ ]:
case = 'Payment feature launch delayed by 5 days. Testing found 3 medium defects. Customer launch is in 2 weeks.'  # Fictional case facts shared with the agents.
print(case)  # Inspect facts before running the workflow.


## Agent conversation

Each role is a short prompt. The coordinator receives reports from earlier roles where the workflow requires a handoff.

In [ ]:
goal = "Complete the defined task using the available evidence."  # Define the target.
plan = ask_agent("Workflow planner", "Plan short steps using only these configured roles: Schedule specialist, Quality specialist, Customer communications specialist, Project manager", case)  # Ask the LLM to plan.
print("PLAN:\n", plan)  # Show the plan before actions.
reports = []  # Store agent messages as observations and handoffs.
report = ask_agent('Schedule specialist', 'Assess schedule impact and propose a recovery action.', case)  # Run the Schedule specialist role.
reports.append(report)  # Pass this message to the next role when needed.
print("\n--- Schedule specialist ---\n", report)  # Make the conversation visible.
report = ask_agent('Quality specialist', 'Assess testing risk and recommend release checks.', case)  # Run the Quality specialist role.
reports.append(report)  # Pass this message to the next role when needed.
print("\n--- Quality specialist ---\n", report)  # Make the conversation visible.
report = ask_agent('Customer communications specialist', 'Draft a factual customer update. Do not promise a date.', case)  # Run the Customer communications specialist role.
reports.append(report)  # Pass this message to the next role when needed.
print("\n--- Customer communications specialist ---\n", report)  # Make the conversation visible.
report = ask_agent('Project manager', 'Combine the specialist reports into an action plan. Identify dependencies and decisions a human must approve.', case + "\n\nPrior agent messages:\n" + "\n".join(reports))  # Run the Project manager role.
reports.append(report)  # Pass this message to the next role when needed.
print("\n--- Project manager ---\n", report)  # Make the conversation visible.
observation = ask_agent("Observer", "Summarize the results and evidence from the agent messages.", case + "\n\nAgent messages:\n" + "\n".join(reports))  # Observe the actions.
print("\nOBSERVATION:\n", observation)
complete = False  # Track whether the goal has been met.
for attempt in range(2):  # Limit the workflow to two review rounds.
    status = ask_agent("Goal checker", "Reply COMPLETE if the goal is met; otherwise NEEDS_REVISION and one specific fix.", goal + "\n\n" + observation)  # Check the result.
    print(f"CHECK {attempt + 1}: {status}")
    if status.strip().upper().startswith("COMPLETE"):
        complete = True
        break  # Stop when the goal checker approves completion.
    observation = ask_agent("Workflow coordinator", "Apply the checker’s fix and return an improved result.", observation + "\n\nChecker: " + status)  # Iterate using feedback.
    print("ITERATION RESULT:\n", observation)
if not complete:
    print("Iteration limit reached. A person should review the result.")


## Plan, act, observe, iterate

The LLM first proposes a plan. The configured agents take actions by making role-specific calls and handing off messages. An observer summarizes the results. A goal checker decides whether to stop or request a revision. The coordinator may iterate up to two times, then asks for human review if the goal remains incomplete.

Master agent delegates to child agents.